In [34]:
import sympy as sp
import numpy as np

from composipy import OrthotropicMaterial, LaminateProperty, PlateStructure

In [2]:
E1 = 60800
E2 = 58250
v12 = 0.07
G12 = 4550
t = 0.21

mat_1 = OrthotropicMaterial(E1, E2, v12, G12, t)

In [3]:
from composipy import LaminateProperty
stacking = [-45, 45, 90, 0, 0, 0, 0, 90, 45, -45]
laminate1 = LaminateProperty(stacking, mat_1)

In [103]:
constraints = {    
    'x0' : ['TX', 'TY', 'TZ', 'RX', 'RY', 'RZ'],
    'xa' : ['TX', 'TY', 'TZ', 'RX', 'RY', 'RZ'],
    'y0' : ['TX', 'TY', 'TZ', 'RX', 'RY', 'RZ'],
    'yb' : ['TX', 'TY', 'TZ', 'RX', 'RY', 'RZ']
    }
panel1 = PlateStructure(laminate1, 360, 360, m=4, n=5, Nxx=-1, constraints=constraints)
panel2 = PlateStructure(laminate1, 360, 360, m=4, n=4, Nxx=-1, constraints=constraints)

In [104]:
K1, _ = panel1.calc_K_KG()
K2, _ = panel2.calc_K_KG()

K1rows, K1col = K1.shape
K2rows, K2col = K2.shape

K012 = np.zeros(K1rows*K2col).reshape(K1rows, K2col)
K021 = np.zeros(K2rows*K1col).reshape(K2rows, K1col)

In [32]:
K = np.vstack([
    np.hstack([K1, K012]),
    np.hstack([K021, K2])
    ])


# Scipy sparse tests

In [89]:
from scipy.sparse import coo_array, bsr_array, csc_array, csr_array, dok_array, dia_array

# https://docs.scipy.org/doc/scipy/reference/sparse.html

In [84]:
# Para ver chave valor de matrix sparsa
# K1s = dok_array(K1)
# K1s.items()

In [88]:
a1 = np.array([[11, 12],
               [21, 22]])

a1 = np.array([[33, 34],
               [43, 44]])

In [ ]:
A = dia_array

In [188]:
def build_assembly_coo_array(matrices):
    '''
    Parameters
    ----------
    matrices : iterable
        Iterable containing 2D numpy array
        ex: [K1, K2, ..., KN]
    
    Returns
    -------
    assembly : coo_array
        An array containing the matrices assembly
        ex: [K1, 0, 0
             0,  K2, 0
             0,  0,  KN]
    '''

    rows = 0
    cols = 0
    offset = 0
    idx = []

    for matrix in matrices:
        r, c = matrix.shape
        rows += r
        cols += c
        ri = np.arange(r) + offset
        ci = np.arange(c) + offset

        idx.append(
            np.array(list(product(ri, ci)))
        )
        offset += r

    data = []
    row = []
    col = []

    for i, matrix in enumerate(matrices):
        m_values = list(
            matrix.reshape(matrix.size))
        
        data.extend(m_values)
        row.extend(list(
            idx[i][::, 0])
        )
        col.extend(list(
            idx[i][::, 1])
        )

    return coo_array((data, (row, col)))



In [194]:
def build_penalty_coo_array_xcte(size, k1, k2):
    pass


In [184]:
a = np.array([[0.1, 0.2],
              [0.3, 0.4]])

b = np.array([[0.5, 0.6],
              [0.7, 0.8]])

c = np.array([[0.9, 0.11, 0.12],
              [0.71, 0.81, 0.55],
              [0.71, 0.81, 0.55]])              

In [191]:
build_assembly_coo_array([a, b, c]).shape

(7, 7)

In [151]:
a = np.arange(10).reshape(2,5)
a

array([[0, 1, 2, 3, 4],
       [5, 6, 7, 8, 9]])

In [153]:
a.reshape(a.size)

array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])

In [142]:
np.arange(3)

array([0, 1, 2])

In [143]:
a = np.array([0, 1, 2, 3]) + 10
a = np.array(list(product(a, a)))
a[::, 1]

array([10, 11, 12, 13, 10, 11, 12, 13, 10, 11, 12, 13, 10, 11, 12, 13])

In [154]:
a = coo_array((2, 3))
a

In [156]:
a.toarray()


array([[0., 0., 0.],
       [0., 0., 0.]])

In [139]:
b = np.array([[1, 2], 
              [4, 5]])
b

array([[1, 2],
       [4, 5]])

In [147]:
k = 0

a = coo_array([[1]])
b = coo_array([[1, 2],
               [3, 4]])

In [148]:
k += a
k += b

ValueError: inconsistent shapes